# Browser-Use Download Functionality Tutorial

## Overview

This tutorial demonstrates how to enable file download capabilities in Browser-Use for remote browser environments. Uses an enhanced browser-use fork with HTTP-based download capabilities.

### Key Features

* **Remote Browser Downloads**: Download files in headless/remote browser environments
* **Agent Download Awareness**: LLM sees download progress and failures in real-time
* **HTTP Client Fallback**: Uses HTTP requests when browser dialogs fail
* **Progress Tracking**: Monitor download progress and handle failures gracefully

## Prerequisites

Before running this tutorial, ensure you have:

* Python 3.11+
* Valid AWS credentials configured
* Git installed

## 1. Installation and Setup

### 1.1 Install Dependencies

In [ ]:
# Install dependencies (no browser-use needed - we'll use enhanced fork)
!pip install bedrock-agentcore boto3 rich --quiet

### 1.2 Setup Enhanced Browser-Use

Instead of installing browser-use from PyPI, use the enhanced version with download functionality.

**Run these commands in your terminal:**

```bash
# Clone the enhanced browser-use fork
git clone https://github.com/rvkandury/browser-use.git
cd browser-use
git checkout feat/remote-browser-support

# Set PYTHONPATH to use enhanced version
export PYTHONPATH=/path/to/your/browser-use:$PYTHONPATH
```

Replace `/path/to/your/browser-use` with the actual path where you cloned the repository.

**Example:**
```bash
export PYTHONPATH=/home/user/browser-use:$PYTHONPATH
```

## 2. Download Example with BrowserClient

### 2.1 Large File Download Example

In [ ]:
from bedrock_agentcore.tools.browser_client import BrowserClient
from browser_use.llm import ChatAnthropicBedrock
from browser_use import Agent
from browser_use import Browser, BrowserProfile
from rich.console import Console
from contextlib import suppress
import asyncio
from boto3.session import Session

console = Console()

async def download_large_file_example():
    """Example of downloading a large file with enhanced browser-use"""
    
    boto_session = Session()
    region = boto_session.region_name
    
    client = BrowserClient(region)
    client.start(viewport={'width': 1920, 'height': 1080})

    ws_url, headers = client.generate_ws_headers()
    browser_session = None

    try:
        # Create browser profile with download enhancement enabled
        browser_profile = BrowserProfile(
            headers=headers,
            timeout=1500000,
            downloads_path="./downloads",
            download_from_remote_browser=True,  # Enable HTTP download fallback
            auto_download_pdfs=True
        )

        browser_session = Browser(
            cdp_url=ws_url,
            browser_profile=browser_profile,
            keep_alive=True
        )
        
        await browser_session.start()
        
        bedrock_chat = ChatAnthropicBedrock(
            model='us.anthropic.claude-3-7-sonnet-20250219-v1:0',
            aws_region='us-west-2'
        )

        task = "Go to https://proof.ovh.net/files and download the 100 MB file. Please wait for the download to finish and tell me when done."

        agent = Agent(
            task=task,
            llm=bedrock_chat,
            browser_session=browser_session,
            llm_timeout=300
        )
        
        result = await agent.run()
        print(f"Download task completed: {result}")
        
        return result

    finally:
        if browser_session:
            with suppress(Exception):
                await browser_session.close()
        client.stop()

# Run the example
await download_large_file_example()

### 2.2 Verify Download Functionality

In [ ]:
# Check if enhanced browser-use is loaded
try:
    import browser_use
    from browser_use import BrowserProfile
    
    print(f"✅ Using browser-use from: {browser_use.__file__}")
    
    # Check if download functionality is available
    profile = BrowserProfile(download_from_remote_browser=True)
    print(f"✅ Download functionality available: download_from_remote_browser={profile.download_from_remote_browser}")
    
except ImportError as e:
    print(f"❌ Enhanced browser-use not found: {e}")
    print("Make sure you've set PYTHONPATH correctly")
except Exception as e:
    print(f"❌ Error: {e}")

### 2.3 Check Downloaded Files

In [ ]:
import os
from pathlib import Path

# Check what files were downloaded
downloads_dir = Path("./downloads")
if downloads_dir.exists():
    downloaded_files = list(downloads_dir.glob("*"))
    print(f"📁 Found {len(downloaded_files)} downloaded files:")
    for file in downloaded_files:
        if file.is_file():
            size_mb = file.stat().st_size / (1024 * 1024)
            print(f"  - {file.name} ({size_mb:.2f} MB)")
else:
    print("📁 Downloads directory not found")

## 3. Configuration Options

The enhanced browser-use adds the following configuration option to `BrowserProfile`:

- **`download_from_remote_browser`** (bool, default: False)
  - `True`: Use HTTP client for downloads (recommended for remote/headless browsers)
  - `False`: Use standard browser download behavior (works for local browsers with file dialogs)

### Best Practices

1. **Always set `download_from_remote_browser=True`** for headless or remote browser scenarios
2. **Specify a `downloads_path`** to control where files are saved
3. **Monitor agent context** - the LLM will see download progress and can make intelligent decisions
4. **Handle failures gracefully** - the agent can retry with different approaches if downloads fail

### Agent Download Awareness

The enhanced agent will see download status in its context:
```
<downloads_in_progress>
Downloading: 100Mio.dat (45%, 12s elapsed)
These downloads are still in progress
</downloads_in_progress>

<failed_downloads>
Failed: file.pdf (Network error) 3m ago
These downloads failed recently
</failed_downloads>
```

## Conclusion

This tutorial demonstrated how to use enhanced Browser-Use with download functionality for remote browser environments. Key takeaways:

- **Enhanced fork enables file downloads** in headless/remote browser scenarios
- **Agent awareness** allows the LLM to monitor download progress and handle failures
- **HTTP client fallback** works when browser file dialogs are not available
- **BrowserClient integration** works seamlessly with bedrock-agentcore
- **Simple setup** with git clone and PYTHONPATH

The enhanced download functionality is particularly useful for:
- Document processing workflows
- Data collection tasks
- Remote browser automation
- Large file downloads with progress monitoring